# Import

In [26]:
import numpy as np
import pandas as pd

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score
from category_encoders import TargetEncoder

from pprint import pprint
from dotenv import load_dotenv

from ucimlrepo import fetch_ucirepo

In [27]:
load_dotenv()
sklearn.set_config(transform_output="pandas")


from src.layers import LinearLayer
from src.activations import ReLU, Sigmoid
from src.losses import BinaryCrossEntropy
from src.optimizers import SGD
from src.models import MLP

# Данные

## Загружаем данные

In [3]:
# Загружаем датасет
mushroom = fetch_ucirepo(id=73)

# Разбиваем на объекты с признаками и объекты с метками
X = mushroom.data.features
y = mushroom.data.targets

## Исследуем данные

In [4]:
# Метаданные
pprint(mushroom.metadata)

{'abstract': 'From Audobon Society Field Guide; mushrooms described in terms '
             'of physical characteristics; classification: poisonous or edible',
 'additional_info': {'citation': None,
                     'funded_by': None,
                     'instances_represent': None,
                     'preprocessing_description': None,
                     'purpose': None,
                     'recommended_data_splits': None,
                     'sensitive_data': None,
                     'summary': 'This data set includes descriptions of '
                                'hypothetical samples corresponding to 23 '
                                'species of gilled mushrooms in the Agaricus '
                                'and Lepiota Family (pp. 500-525).  Each '
                                'species is identified as definitely edible, '
                                'definitely poisonous, or of unknown edibility '
                                'and not recommended. 

In [5]:
# Информация о переменных
mushroom.variables

,name,role,type,demographic,description,units,missing_values
0,poisonous,Target,Categorical,None,None,None,no
1,cap-shape,Feature,Categorical,None,"bell=b,conical=c,convex=x,flat=f, knobbed=k,su...",None,no
2,cap-surface,Feature,Categorical,None,"fibrous=f,grooves=g,scaly=y,smooth=s",None,no
3,cap-color,Feature,Binary,None,"brown=n,buff=b,cinnamon=c,gray=g,green=r, pink...",None,no
4,bruises,Feature,Categorical,None,"bruises=t,no=f",None,no
5,odor,Feature,Categorical,None,"almond=a,anise=l,creosote=c,fishy=y,foul=f, mu...",None,no
6,gill-attachment,Feature,Categorical,None,"attached=a,descending=d,free=f,notched=n",None,no
7,gill-spacing,Feature,Categorical,None,"close=c,crowded=w,distant=d",None,no
8,gill-size,Feature,Categorical,None,"broad=b,narrow=n",None,no
9,gill-color,Feature,Categorical,None,"black=k,brown=n,buff=b,chocolate=h,gray=g, gre...",None,no


In [6]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   cap-shape                 8124 non-null   object
 1   cap-surface               8124 non-null   object
 2   cap-color                 8124 non-null   object
 3   bruises                   8124 non-null   object
 4   odor                      8124 non-null   object
 5   gill-attachment           8124 non-null   object
 6   gill-spacing              8124 non-null   object
 7   gill-size                 8124 non-null   object
 8   gill-color                8124 non-null   object
 9   stalk-shape               8124 non-null   object
 10  stalk-root                5644 non-null   object
 11  stalk-surface-above-ring  8124 non-null   object
 12  stalk-surface-below-ring  8124 non-null   object
 13  stalk-color-above-ring    8124 non-null   object
 14  stalk-color-below-ring  

In [7]:
print('Процент Nan данных для stalk-root:', 1 - 5644/8124)

Процент Nan данных для stalk-root: 0.30526834071885767


In [8]:
pd.concat([X, y], axis=1).describe()

,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,poisonous
count,8124,8124,8124,8124,8124,8124,8124,8124,8124,8124,...,8124,8124,8124,8124,8124,8124,8124,8124,8124,8124
unique,6,4,10,2,9,2,2,2,12,2,...,9,9,1,4,3,5,9,6,7,2
top,x,y,n,f,n,f,c,b,b,t,...,w,w,p,w,o,p,w,v,d,e
freq,3656,3244,2284,4748,3528,7914,6812,5612,1728,4608,...,4464,4384,8124,7924,7488,3968,2388,4040,3148,4208


In [9]:
for col in X.columns:
    print(f"{col}: {X[col].unique()}")

cap-shape: ['x' 'b' 's' 'f' 'k' 'c']
cap-surface: ['s' 'y' 'f' 'g']
cap-color: ['n' 'y' 'w' 'g' 'e' 'p' 'b' 'u' 'c' 'r']
bruises: ['t' 'f']
odor: ['p' 'a' 'l' 'n' 'f' 'c' 'y' 's' 'm']
gill-attachment: ['f' 'a']
gill-spacing: ['c' 'w']
gill-size: ['n' 'b']
gill-color: ['k' 'n' 'g' 'p' 'w' 'h' 'u' 'e' 'b' 'r' 'y' 'o']
stalk-shape: ['e' 't']
stalk-root: ['e' 'c' 'b' 'r' nan]
stalk-surface-above-ring: ['s' 'f' 'k' 'y']
stalk-surface-below-ring: ['s' 'f' 'y' 'k']
stalk-color-above-ring: ['w' 'g' 'p' 'n' 'b' 'e' 'o' 'c' 'y']
stalk-color-below-ring: ['w' 'p' 'g' 'b' 'n' 'e' 'y' 'o' 'c']
veil-type: ['p']
veil-color: ['w' 'n' 'o' 'y']
ring-number: ['o' 't' 'n']
ring-type: ['p' 'e' 'l' 'f' 'n']
spore-print-color: ['k' 'n' 'u' 'h' 'w' 'r' 'o' 'y' 'b']
population: ['s' 'n' 'a' 'v' 'y' 'c']
habitat: ['u' 'g' 'm' 'd' 'p' 'w' 'l']


## Подготавливаем данные

In [10]:
# Индентифицируем метки классов
labels = {
    0: 'edible',
    1: 'poisonous'
}

In [11]:
# Инструменты для предобработки
label_encoder = LabelEncoder()
target_encoder = TargetEncoder(cols=['cap-shape', 'cap-color', 'odor', 'gill-color', 'stalk-color-above-ring',
                                     'stalk-color-below-ring', 'spore-print-color' ,'population', 'habitat'])
minmax_scaler = MinMaxScaler()

In [12]:
# Кодируем метки
y_prep = (y == 'p').astype(int)

X_prep = X.copy()
# Кодируем бинарные признаки
X_prep['bruises'] = (X_prep['bruises'] == 'f').astype(int)
X_prep['gill-attachment'] = (X_prep['gill-attachment'] == 'f').astype(int)
X_prep['gill-spacing'] = (X_prep['gill-spacing'] == 'c').astype(int)
X_prep['gill-size'] = (X_prep['gill-size'] == 'b').astype(int)
X_prep['stalk-shape'] = (X_prep['stalk-shape'] == 't').astype(int)
# Удаляем ненужные признаки
X_prep = X_prep.drop('veil-type', axis=1)
# Кодируем признаки с менее 6 уникальными значениями
X_prep['cap-surface'] = label_encoder.fit_transform(X_prep['cap-surface'])
X_prep['stalk-root'] = label_encoder.fit_transform(X_prep['stalk-root'])
X_prep['stalk-surface-above-ring'] = label_encoder.fit_transform(X_prep['stalk-surface-above-ring'])
X_prep['stalk-surface-below-ring'] = label_encoder.fit_transform(X_prep['stalk-surface-below-ring'])
X_prep['veil-color'] = label_encoder.fit_transform(X_prep['veil-color'])
X_prep['ring-number'] = label_encoder.fit_transform(X_prep['ring-number'])
X_prep['ring-type'] = label_encoder.fit_transform(X_prep['ring-type'])
# Кодируем признаки с более 6 уникальными значениями
target_encoder.fit(X_prep, y_prep)
X_prep = target_encoder.transform(X_prep)

In [13]:
# Нормализуем данные
minmax_scaler.fit(X_prep)
X_prep = minmax_scaler.transform(X_prep)
X_prep.describe()

,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-surface-above-ring,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-color,ring-number,ring-type,spore-print-color,population,habitat
count,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,...,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000,8124.000000
mean,0.608077,0.609224,0.457304,0.584441,0.481643,0.974151,0.838503,0.690793,0.481342,0.567208,...,0.525029,0.534548,0.481254,0.481032,0.655178,0.534712,0.572994,0.469327,0.683776,0.547064
std,0.195081,0.409958,0.247560,0.492848,0.484842,0.158695,0.368011,0.462195,0.339478,0.495493,...,0.207153,0.225325,0.261088,0.255499,0.080890,0.135532,0.450418,0.387048,0.345479,0.249614
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.583696,0.000000,0.353642,0.000000,0.034014,1.000000,1.000000,0.000000,0.156659,0.000000,...,0.333333,0.333333,0.383513,0.383212,0.666667,0.500000,0.000000,0.088898,0.418287,0.390988
50%,0.583696,0.666667,0.371153,1.000000,0.034014,1.000000,1.000000,1.000000,0.428817,1.000000,...,0.666667,0.666667,0.383513,0.383212,0.666667,0.500000,0.500000,0.754144,0.536924,0.457141
75%,0.626509,1.000000,0.693950,1.000000,1.000000,1.000000,1.000000,1.000000,0.721244,1.000000,...,0.666667,0.666667,0.692308,0.692308,0.666667,0.500000,1.000000,0.754144,1.000000,0.807540
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [14]:
# Кодируем метки
y_prep = (y == 'p').astype(int)

X_prep = X.copy()
# Кодируем бинарные признаки
X_prep['bruises'] = (X_prep['bruises'] == 'f').astype(int)
X_prep['gill-attachment'] = (X_prep['gill-attachment'] == 'f').astype(int)
X_prep['gill-spacing'] = (X_prep['gill-spacing'] == 'c').astype(int)
X_prep['gill-size'] = (X_prep['gill-size'] == 'b').astype(int)
X_prep['stalk-shape'] = (X_prep['stalk-shape'] == 't').astype(int)
# Удаляем ненужные признаки
X_prep = X_prep.drop('veil-type', axis=1)
# Остальные кодируем с помощью one-hot-encoding
X_prep = pd.get_dummies(X_prep, columns=['cap-surface', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'veil-color', 'ring-number', 'ring-type',
                                         'cap-shape', 'cap-color', 'odor', 'gill-color', 'stalk-color-above-ring', 'stalk-color-below-ring', 'spore-print-color' ,'population', 'habitat']).astype(int)

X_prep

,bruises,gill-attachment,gill-spacing,gill-size,stalk-shape,cap-surface_f,cap-surface_g,cap-surface_s,cap-surface_y,stalk-root_b,...,population_s,population_v,population_y,habitat_d,habitat_g,habitat_l,habitat_m,habitat_p,habitat_u,habitat_w
0,0,1,1,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,1,0
1,0,1,1,1,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,1,1,1,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0,1,1,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,1,0
4,1,1,0,1,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,1,0,1,1,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
8120,1,0,1,1,0,0,0,1,0,0,...,0,1,0,0,0,1,0,0,0,0
8121,1,0,1,1,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
8122,1,1,1,0,1,0,0,0,1,0,...,0,1,0,0,0,1,0,0,0,0


In [15]:
# Кодируем метки
y_prep = (y == 'p').astype(int)

X_prep = X.copy()
k = X_prep.keys()
for i in k:
    le = LabelEncoder()
    n = str(i) + "_n"
    X_prep[n] = le.fit_transform(X_prep[i])

for i in k:
    del X_prep[i]

X_prep.head()

,cap-shape_n,cap-surface_n,cap-color_n,bruises_n,odor_n,gill-attachment_n,gill-spacing_n,gill-size_n,gill-color_n,stalk-shape_n,...,stalk-surface-below-ring_n,stalk-color-above-ring_n,stalk-color-below-ring_n,veil-type_n,veil-color_n,ring-number_n,ring-type_n,spore-print-color_n,population_n,habitat_n
0,5,2,4,1,6,1,0,1,4,0,...,2,7,7,0,2,1,4,2,3,5
1,5,2,9,1,0,1,0,0,4,0,...,2,7,7,0,2,1,4,3,2,1
2,0,2,8,1,3,1,0,0,5,0,...,2,7,7,0,2,1,4,3,2,3
3,5,3,8,1,6,1,0,1,5,0,...,2,7,7,0,2,1,4,2,3,5
4,5,2,3,0,5,1,1,0,4,1,...,2,7,7,0,2,1,0,3,0,1


In [16]:
# Разделяем на выборки
X_train, X_test, y_train, y_test = map(np.array, train_test_split(X_prep, y_prep, test_size=0.2, random_state=42, shuffle=True, stratify=y_prep))
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, shuffle=True, stratify=y_train)

# Модель

In [28]:
# Задаём архитектуру модели
arch_model = [
    LinearLayer(in_features=X_train.shape[1], out_features=50, bias=True, activation_function=Sigmoid()),
    LinearLayer(in_features=50, out_features=1, bias=True, activation_function=Sigmoid()),
    # LinearLayer(in_features=8, out_features=1, bias=True, activation_function=Sigmoid())
]
# Инициализируем Loss и оптимизатор
loss = BinaryCrossEntropy(labels=labels)
optimizer = SGD(batch_size=4, lr=0.001)
# Создаём модель
model = MLP(
    arch_model,
    loss,
    optimizer
)

In [29]:
def postprocess(inputs: np.ndarray) -> np.ndarray:
    return (inputs > 0.5).astype(int)

# Обучение

In [19]:
# # Параметры
# np.random.seed(42)
# num_inputs = 5
# num_classes = 3
# learning_rate = 0.1
# num_samples = 10

# # Входы и метки
# X = np.random.randn(num_samples, num_inputs)
# y = np.random.randint(0, num_classes, size=num_samples)

# # Инициализация весов и смещений
# W = np.random.randn(num_inputs, num_classes)
# b = np.zeros(num_classes)

# # Softmax
# def softmax(z):
#     exp_z = np.exp(z - np.max(z))
#     return exp_z / np.sum(exp_z)

# # One-hot encoding
# Y_one_hot = np.zeros((num_samples, num_classes))
# for n in range(num_samples):
#     Y_one_hot[n, y[n]] = 1

# # Поэлементное обновление весов
# for n in range(num_samples):
#     # Вычисляем z и a для примера n
#     z = np.zeros(num_classes)
#     for j in range(num_classes):
#         for i in range(num_inputs):
#             z[j] += X[n, i] * W[i, j]
#         z[j] += b[j]
    
#     a = softmax(z)
    
#     # dL/da = a - y_one_hot (для кросс-энтропии, но оставляем явно)
#     dL_da = a - Y_one_hot[n]
    
#     # Для каждого нейрона вычисляем da/dz в виде вектора
#     for j in range(num_classes):          # нейрон
#         da_dz = np.zeros(num_classes)
#         for k in range(num_classes):
#             if j == k:
#                 da_dz[k] = a[j] * (1 - a[j])
#             else:
#                 da_dz[k] = -a[j] * a[k]
        
#         print(da_dz)
#         # dL/dz_j = sum_k (dL/da_k * da_k/dz_j)
#         dL_dz = 0
#         for k in range(num_classes):
#             dL_dz += dL_da[k] * da_dz[k]
        
#         # Обновление весов по входам
#         for i in range(num_inputs):
#             dz_dw = X[n, i]                   # dz_j/dW_ij = x_i
#             dL_dW = dL_dz * dz_dw
#             W[i, j] -= learning_rate * dL_dW
        
#         # Обновление смещения
#         b[j] -= learning_rate * dL_dz
        
#         break
#     break

# # print("Обновленные веса:\n", W)
# # print("Обновленные смещения:\n", b)


# jacobian = np.zeros((3, 3))
# for i in range(3):
#     for j in range(3):
#         if i == j:
#             jacobian[i, j] = outputs[i] * (1 - outputs[j])
#         else:
#             jacobian[i, j] = -outputs[i] * outputs[j]

# jacobian

In [30]:
# Задаём параметры обучения и запускаем его
n_epochs = 15
verbose_n_batch_multiple = 10
model.train_model(X_train, y_train.ravel(), X_val, y_val.ravel(), n_epochs, postprocess, accuracy_score, verbose_n_batch_multiple)

Epoch 1 (0/5199):                           BinaryCrossEntropy = 4.029                           accuracy_score = 0.482
Epoch 1 (40/5199):                           BinaryCrossEntropy = 5.037                           accuracy_score = 0.482
Epoch 1 (80/5199):                           BinaryCrossEntropy = 4.701                           accuracy_score = 0.482
Epoch 1 (120/5199):                           BinaryCrossEntropy = 5.037                           accuracy_score = 0.482
Epoch 1 (160/5199):                           BinaryCrossEntropy = 5.238                           accuracy_score = 0.482
Epoch 1 (200/5199):                           BinaryCrossEntropy = 5.037                           accuracy_score = 0.482
Epoch 1 (240/5199):                           BinaryCrossEntropy = 4.893                           accuracy_score = 0.482
Epoch 1 (280/5199):                           BinaryCrossEntropy = 4.785                           accuracy_score = 0.482
Epoch 1 (320/5199):         

In [34]:
# Проверка на предсказаниях
y_pred_labels = model.predict(X_test, postprocess).ravel()
acc = accuracy_score(y_test.ravel(), y_pred_labels)
print("Accuracy:", acc)

Accuracy: 0.48184615384615387


In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Допустим у нас данные
# X_train: [n_samples, n_features]
# y_train: [n_samples,]

# Датасет и загрузчик
dataset = TensorDataset(torch.Tensor(X_train), torch.Tensor(y_train))
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Определяем MLP модель
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, 50),
            nn.Sigmoid(),
            nn.Linear(50, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.model(x)

# Создаем модель
model = MLP(X_train.shape[1])

# Оптимизатор и функция потерь
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)

# Цикл обучения
n_epochs = 15
for epoch in range(n_epochs):
    for X_batch, y_batch in dataloader:
        # forward
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        
        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6700
Epoch 2, Loss: 0.6125
Epoch 3, Loss: 0.5000
Epoch 4, Loss: 0.6212
Epoch 5, Loss: 0.3739
Epoch 6, Loss: 0.6721
Epoch 7, Loss: 0.3468
Epoch 8, Loss: 0.3204
Epoch 9, Loss: 0.2214
Epoch 10, Loss: 0.1623
Epoch 11, Loss: 0.2650
Epoch 12, Loss: 0.6937
Epoch 13, Loss: 0.2102
Epoch 14, Loss: 0.9393
Epoch 15, Loss: 0.1611


In [25]:
# Проверка на предсказаниях
with torch.no_grad():
    y_pred_probs = model(torch.Tensor(X_test))
    y_pred_labels = (y_pred_probs > 0.5).int().squeeze()
    acc = accuracy_score(y_test.ravel(), y_pred_labels)
    print("Accuracy:", acc)

Accuracy: 0.8609230769230769


# Тестирование

In [15]:
def _to_one_hot(y_true: np.ndarray, n_labels: int | None = None) -> np.ndarray:
    """
    Преобразует список с метками в one-hot encoding список.

    :param y_true: Массив меток классов shape = [n_samples,].
    :type y_true: np.ndarray
    :param n_labels: Количество классов (если None, берётся max + 1).
    :type n_labels: int | None

    :return: Список закодированными метками класса размера [n_samples, n_labels]
    :rtype: np.ndarray
    """
    if n_labels is None:
        n_labels = np.max(y_true) + 1
    
    return np.eye(n_labels)[y_true].squeeze()

def _expand_binary_probs(y_pred: np.ndarray) -> np.ndarray:
    """
    Преобразует список вероятностей одного класса [n_samples, 1]
    в вероятности двух классов [n_samples, 2].

    :param y_pred: массив вероятностей первого класса (размер: [n_samples, 1]).
    :type y_pred: np.ndarray

    :return: Массив вероятностей [p, 1 - p] (размер: [n_samples, 2]).
    :rtype: np.ndarray
    """
    # Если размер [n_samples,], то переводим в [n_samples, 1]
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 1)

    # Если и так уже две вероятности, то менять ничего не надо
    if y_pred.shape[1] == 2:
        return y_pred

    p = y_pred[:, 0]
    return np.column_stack([1 - p, p])

In [26]:
yt = y_train[:3]
yp = np.array([
    [0.3],
    [0.9],
    [0.1]
])

tyt = _to_one_hot(yt, 2)
typ = _expand_binary_probs(yp)

losses = np.mean(-tyt*np.log(typ), axis=0)

print(losses, '\n')

# Если бинарная классификация и выходной слой отдаёт 1 значение, то подгоняем формат
if yp.ndim == 1 or yp.shape[1] == 1:
    losses = np.array([np.mean(losses)])
    print(losses)

np.mean(losses)

[0.03512017 0.43644444] 

[0.23578231]


np.float64(0.23578230594026478)

In [60]:
tX = X_train[:3]

tW1 = np.random.rand(tX.shape[1])
tW2 = np.random.rand(tX.shape[1])

Z1 = tX@tW1
Z2 = tX@tW2

Z = np.array([Z1, Z2])
A = 1 / (1 + np.exp(-Z))
output = A

print(output[0])

[0.99962978 0.99977777 0.99356163]
